In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [2]:
import math
import numpy as np
import torch

import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

## 模型

In [3]:
from models import ViT, SplitedViT, load_partial_weight

/home/chengguoliang/anaconda3/envs/pt2.3/lib/python3.9/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## 构建数据集

In [4]:
from datasets_fl import get_classification_dataset

In [5]:
class DatasetSplit(Dataset):
    """An abstract Dataset class wrapped around Pytorch Dataset class.
    """
    def __init__(self, dataset, idxs, indi_err=None):
        self.dataset = dataset
        self.idxs = [int(i) for i in idxs]
        self.indi_err = indi_err
        self.targets = [self.dataset[idx][1] for idx in self.idxs]
        self.num_classes = len(set(dataset.targets))
                         
    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, item):
        image, label = self.dataset[self.idxs[item]]
        if self.indi_err is not None: 
            if self.indi_err[item] == True:
                label = (label+random.randint(1,9))%10
        return image.clone().detach(), torch.tensor(label)

## 训练函数

In [6]:
import math

from torch.optim.lr_scheduler import LambdaLR

class WarmupCosineSchedule(LambdaLR):
    """ Linear warmup and then cosine decay.
        Linearly increases learning rate from 0 to 1 over `warmup_steps` training steps.
        Decreases learning rate from 1. to 0. over remaining `t_total - warmup_steps` steps following a cosine curve.
        If `cycles` (default=0.5) is different from default, learning rate follows cosine function after warmup.
    """
    def __init__(self, optimizer, warmup_steps, t_total, cycles=.5, last_epoch=-1):
        self.warmup_steps = warmup_steps
        self.t_total = t_total
        self.cycles = cycles
        super(WarmupCosineSchedule, self).__init__(optimizer, self.lr_lambda, last_epoch=last_epoch)

    def lr_lambda(self, step):
        if step < self.warmup_steps:
            return float(step) / float(max(1.0, self.warmup_steps))
        # progress after warmup
        progress = float(step - self.warmup_steps) / float(max(1, self.t_total - self.warmup_steps))
        return max(0.0, 0.5 * (1. + math.cos(math.pi * float(self.cycles) * 2.0 * progress)))

In [7]:
class Aggregator:                                                              
    def __init__(self, glo_model, device):
        self.device = device
        self.glo_model = glo_model

    @torch.no_grad()    
    def agg(self, local_model, n0):
        total_n = self.n + n0       
        for i, local_p in enumerate(local_model.parameters()):
            self.average_model[i].mul_(self.n / total_n)
            local_p.mul_(n0 / total_n)   
            self.average_model[i].add_(local_p)
        self.n = total_n  
    
    def glo_model_update(self):
        for j, global_p in enumerate(self.glo_model.parameters()):
            global_p.data = self.average_model[j]
        return self.glo_model
    
    def zero_n(self):
        self.n = 0
        self.average_model = []
        for para in self.glo_model.parameters():
            average_model_zero = torch.zeros_like(para).to(self.device)
            self.average_model.append(average_model_zero)
            

class Cloud_Server:
    def __init__(self, cloud_loras, clients, aggregator, cloud_aggregator, device="cuda"):
        self.device = device
        self.cloud_loras = cloud_loras
        self.optimizers = [torch.optim.SGD(
                c_lora.parameters(),
                lr=Client.lr,
                momentum=0.9
        ) for c_lora in self.cloud_loras]
        self.clients = clients
        self.aggregator = aggregator
        self.aggregator.zero_n()
        self.cloud_aggregator = cloud_aggregator
        self.cloud_aggregator.zero_n()
        
    def collect_lora(self, client_id, lora, coefficient):
        self.cloud_aggregator.agg(self.cloud_loras[client_id], coefficient)
        self.aggregator.agg(lora, coefficient)
        
    def agg_loras(self):
        glo_lora = self.aggregator.glo_model_update()
        for client in self.clients:
            client.device_lora.load_state_dict(glo_lora.state_dict())
        self.aggregator.zero_n()
        glo_cloud_lora = self.cloud_aggregator.glo_model_update()
        for c_lora in self.cloud_loras:
            c_lora.load_state_dict(glo_cloud_lora.state_dict())
        self.cloud_aggregator.zero_n()
        return glo_lora, glo_cloud_lora

class Client:
    train_set = None
    batch_size = None
    local_epoch = None
    lr = None
    lr_decay = None
    def __init__(self, train_idx, client_id, device_lora, device="cuda"):
        self.client_id = client_id
        self.trainloader = DataLoader(DatasetSplit(self.train_set, train_idx), 
                                      batch_size=self.batch_size, shuffle=True, num_workers=2)
        self.num_data = len(train_idx)
        self.device = device
        self.device_lora = device_lora
        self.optimizer = torch.optim.SGD(
                self.device_lora.parameters(), 
                lr=self.lr,
                momentum=0.9
            )

# lora微调

In [8]:
import math
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random as rn
from copy import deepcopy
from torchvision import datasets, transforms
##
from datasets_fl import get_classification_dataset

In [9]:
@torch.no_grad()
def eval_test(model, testloader, device='cuda'):
    total_num_data = len(testloader.dataset)
    # model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= total_num_data
    print('\nTest: average loss: {:.4f}, accuracy: {}/{} ({:.2f}%)'.format(
        test_loss, correct, total_num_data,
        100. * correct / total_num_data))
    return 100. *correct/total_num_data, test_loss

## 联邦学习函数

In [10]:
from compression import Compressor
from models import LoRA

In [11]:
def get_cps_fw_hook(compressor, cps_list):
    def fw_hook(module, fea_in, fea_out):
        cps_fea_out, size = compressor.compress(fea_out)
        cps_list.append(size)
        return fea_out + (cps_fea_out-fea_out).detach()
    return fw_hook

def get_cps_bw_hook(compressor, cps_list):
    def bw_hook(module, grad_in, grad_out):
        cps_grad_in, size = compressor.compress(grad_in[0])
        cps_list.append(size)
        return [cps_grad_in]
    return bw_hook

In [13]:
def fedlearn(total_cr, fw_cps, bw_cps, sparsity, bit_width, iid=True, seed=0):
    data_dir = '/home/chengguoliang/dataset/CIFAR100'
    num_clients = 8
    num_cloud_servers = 1
    device = 'cuda'
    rn.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_dataset, test_dataset, user_groups, num_classes, fig_size = get_classification_dataset(
        "CIFAR100", data_dir, num_clients, iid=iid, alpha=0.5
    )
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, num_workers=8)
    
    Client.train_set = train_dataset
    Client.batch_size = 64
    Client.lr = 1e-3
    Client.local_epoch = 1
    Client.lr_decay = 0.998
    lr_scheduler = None

    ### loading the training model
    # vit = ViT(
    #     image_size = 224,
    #     patch_size = 16,
    #     num_classes = 200,
    #     dim = 768,
    #     depth = 12,
    #     heads = 12,
    #     mlp_dim = 3072,
    #     dropout = 0.1,
    #     emb_dropout = 0.0,
    #     qkv_bias= True
    # ).to(device)
    
    vit = ViT(
        image_size = 224,
        patch_size = 16,
        num_classes = 200,
        dim = 1024,
        depth = 24,
        heads = 16,
        mlp_dim = 4096,
        dropout = 0.1,
        emb_dropout = 0.1,
        qkv_bias= True
    ).to(device)
    
    # load weight
    ckpt_path = './pretrained_weights/models--vision-transformer/large_p16_224_backbone.pth'
    print('Loading Weights from \'{}\''.format(ckpt_path))
    print()
    weight = torch.load(ckpt_path)
    load_partial_weight(vit, weight)
    
    splitedvit = SplitedViT(vit)
    splited_model = splitedvit.to(device)    
    splited_model.requires_grad_(False)
    
    ### print information of local trianing
    info = {}
    num_data = np.zeros(len(user_groups))
    for i in range(len(user_groups)):
        num_data[i] = len(user_groups[i])
    num_param = 0
    for param in splited_model.parameters():
        num_param += param.numel()
    print("***************************************************************************************************")
    
    key_words = ('fn.fn.net.0', 'fn.fn.net.3', 'fn.fn.to_qkv', '.fn.fn.to_out.0', 'vit.mlp_head.1', 'to_patch_embedding.0')
    
    
    device_lora = LoRA(key_words, rank=16)
    device_lora.register_lora(splited_model.devices_layers)
    device_lora.remove_lora(splited_model.devices_layers)

    cloud_server_lora = LoRA(key_words, rank=16)
    cloud_server_lora.register_lora(splited_model.cloud_server_layers)
    cloud_server_lora.remove_lora(splited_model.cloud_server_layers)
    
    ### initilize a set of clients and the servers
    clients = []
    for idx in range(num_clients):
        clients.append(Client(user_groups[idx], client_id=idx, device_lora=deepcopy(device_lora)))
    clients_ids = [idx for idx in range(num_clients)]
    
    client_aggregator = Aggregator(deepcopy(device_lora), device)
    cloud_aggregator = Aggregator(deepcopy(cloud_server_lora), device)
    cloud_server = Cloud_Server(
        [deepcopy(cloud_server_lora) for i in range(num_clients)], clients,
        client_aggregator, cloud_aggregator, device
    )
    print('successfuly init the clients and servers')
    
    ### initilize the mixed (sparsification, quantization, coding) compressor
    fw_compressor = Compressor()
    bw_compressor = Compressor()
    fw_cps_size = []
    bw_cps_size = []
    if fw_cps:
        print('successfuly add forward_hook into module')
        fw_hook_handle = splited_model.device_blocks[-1][-1].register_forward_hook(
            get_cps_fw_hook(fw_compressor, fw_cps_size)
        )
        fw_compressor.set(sparsity, bit_width)
    if bw_cps:
        print('successfuly add backward_hook into module')
        bw_hook_handle = splited_model.cloud_server_blocks[0][0].register_full_backward_hook(
            get_cps_bw_hook(bw_compressor, bw_cps_size)
        )
        bw_compressor.set(sparsity, bit_width)
    acc_list = []
    test_loss_list = []
    criterion = nn.CrossEntropyLoss()
    
    for cr in range(total_cr): 
        print("##########################################################")
        c_grads = []
        num_data = []
        num_done = 0
        splited_model.train()
        for c_id in clients_ids:
            num_done+=1
            client = clients[c_id]
            
            optimizers = [client.optimizer, cloud_server.optimizers[client.client_id]]
            
            num_processing_data = 0
            for batch_idx, (images, labels) in enumerate(client.trainloader):
                num_processing_data += len(images)
                images, labels = images.to(device), labels.to(device)
                for optim in optimizers:
                    optim.zero_grad()

                cps_smash_data = splited_model.device_model_forward(images, lora=client.device_lora)
                ori_size = cps_smash_data.numel()*32
                output = splited_model.cloud_server_model_forward(
                    cps_smash_data, lora=cloud_server.cloud_loras[client.client_id]
                )

                loss = criterion(output, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(client.device_lora.parameters(), 10)
                nn.utils.clip_grad_norm_(cloud_server.cloud_loras[client.client_id].parameters(), 10)
                
                fw_cps_rate = fw_cps_size[0]/ori_size if fw_cps else 1
                bw_cps_rate = bw_cps_size[0]/ori_size if bw_cps else 1
                if fw_cps:
                    fw_cps_size.clear()
                if bw_cps:
                    bw_cps_size.clear()
                
                for optim in optimizers:
                    optim.step()
                progress = math.ceil((batch_idx+1) / len(client.trainloader) * 50)
                print("\rTrain cr %d(%d-th client), loss=%.5f: %d/%d, fw/bw_cpr_rate=(%.6f, %.6f), [%-51s] %d%%" %
                      (cr, c_id, loss.item(), num_processing_data, len(client.trainloader.dataset), 
                       fw_cps_rate, bw_cps_rate,
                       '-' * progress + '>', progress*2 ), end='')
            print()

        for client in cloud_server.clients:
            cloud_server.collect_lora(client.client_id, client.device_lora, client.num_data)
        glo_device_lora, glo_cloud_lora = cloud_server.agg_loras()
        
        def test_model(images):
            splited_model.eval()
            smash_data1 = splited_model.device_model_forward(images, lora=glo_device_lora)
            output = splited_model.cloud_server_model_forward(smash_data1, lora=glo_cloud_lora)
            return output
            
        acc, test_loss = eval_test(test_model, test_loader, device)
        
        acc_list.append(acc) 
        test_loss_list.append(test_loss)
        
        save_dict = {
            "acc_list":acc_list, 
            "test_loss_list":test_loss_list, 
        }
        torch.save(
            save_dict, './hist/large/cifar100/EiffSplitedFL_%s_vit_large(fw_cps=%s,bw_cps=%s,s=%.2f,q=%d)_log_seed=%d.pth'%(
                'iid' if iid else 'noniid', fw_cps, bw_cps, sparsity, bit_width, seed)
        )

In [ ]:
for seed in [0]:
    for iid_state in [False]:
        fedlearn(20, fw_cps=True, bw_cps=False, sparsity=0., bit_width=32, iid=iid_state, seed=seed)

Files already downloaded and verified
Files already downloaded and verified
Loading Weights from './pretrained_weights/models--vision-transformer/large_p16_224_backbone.pth'

match the state_dict from loading weights, matching rate is  100.0 %
***************************************************************************************************
successfuly init the clients and servers
successfuly add forward_hook into module
##########################################################
Train cr 0(0-th client), loss=1.46316: 6544/6544, fw/bw_cpr_rate=(1.031251, 1.000000), [-------------------------------------------------->] 100%
Train cr 0(1-th client), loss=0.68241: 6254/6254, fw/bw_cpr_rate=(1.031250, 1.000000), [-------------------------------------------------->] 100%
Train cr 0(2-th client), loss=0.03459: 5762/5762, fw/bw_cpr_rate=(1.031255, 1.000000), [-------------------------------------------------->] 100%
Train cr 0(3-th client), loss=0.65112: 6505/6505, fw/bw_cpr_rate=(1.031250, 1